# Fig S7 — AFR and EUR studies with matched and cross-population LD (data processing)

From `28.FigS7.AFR_EUR_Cross_Reference`: 100% AFR and 100% EUR studies (n = 800), each with matched and cross-population LD (AFR from 1000 Genomes; EUR from UK Biobank).

**Outputs:**
- `figure_data/FigS7.AFR_EUR_Cross_Reference_Fallback.rds` — SER fallback fraction by study × LD × setting (S7a)
- `figure_data/FigS7.AFR_EUR_Cross_Reference_Power_FDR.rds` — power / FDR of SuSiE-RSS and the full SuSiE-rRSS pipeline (SER model when flagged), bootstrap 95% CI (S7b)

The simulations use nominal nh² = 20 / 40 / 80 / 120, with one to three causal variants.

In [2]:
source("../../R/utils.R")

    data_name <- "FigS7.AFR_EUR_Cross_Reference"
n_GWAS    <- 800        # NCP = 800 x h2g
res_file  <- "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/AFR_EUR_cross_ref_fine/summary/result_cross_ref_summary.rds"

res <- readRDS(res_file) %>%
  mutate(ld     = str_extract(method, "(AFR|EUR)(?=_LD)"),
         model  = if_else(str_starts(method, "rrss_"), "SuSiE-rRSS", "SuSiE-RSS"),
         is_ser = str_ends(method, "_SER"),
         match  = if_else(study == ld, "Matched LD", "Cross-population LD"))

## failed fits (counted as analyses without a credible set below; listed here)
res %>% filter(fit_failed) %>% count(study, method)

study,method,n
<chr>,<chr>,<int>


## Fig S7a: SER fallback

In [3]:
## ---- Fig S7a: SER fallback, every analysis in the denominator (incl. those without a credible set) ----
fallback <- res %>%
  filter(model == "SuSiE-rRSS", !is_ser) %>%
  group_by(study, ld, match) %>%
  summarise(n_analyses = n(), n_fallback = sum(R_reliability_flag %in% TRUE), .groups = "drop") %>%
  mutate(pct_fallback = round(100 * n_fallback / n_analyses, 1),
         text = sprintf("%.1f%% (%d/%d)", pct_fallback, n_fallback, n_analyses))
fallback

## by setting, for the panel
fallback_by_setting <- res %>%
  filter(model == "SuSiE-rRSS", !is_ser) %>%
  group_by(study, ld, match, h2g, causal) %>%
  summarise(n_analyses = n(), n_fallback = sum(R_reliability_flag %in% TRUE), .groups = "drop") %>%
  mutate(fallback = n_fallback / n_analyses, ncp = n_GWAS * h2g)
save_data(fallback_by_setting, paste0(data_name, "_Fallback"))

## Results text (l.203-204)
with(fallback, cat(sprintf(
  "With AFR study statistics and EUR LD, the pipeline assigned %s of simulations to SER fallback, compared with %s with AFR LD.\n",
  text[study == "AFR" & ld == "EUR"], text[study == "AFR" & ld == "AFR"])))

study,ld,match,n_analyses,n_fallback,pct_fallback,text
<chr>,<chr>,<chr>,<int>,<int>,<dbl>,<chr>
AFR,AFR,Matched LD,5745,106,1.8,1.8% (106/5745)
AFR,EUR,Cross-population LD,5745,5426,94.4,94.4% (5426/5745)
EUR,AFR,Cross-population LD,5952,1302,21.9,21.9% (1302/5952)
EUR,EUR,Matched LD,5952,0,0.0,0.0% (0/5952)


With AFR study statistics and EUR LD, the pipeline assigned 94.4% (5426/5745) of simulations to SER fallback, compared with 1.8% (106/5745) with AFR LD.


## Fig S7b: power / FDR

In [4]:
## ---- Fig S7b: power / FDR, SuSiE-RSS vs. the full SuSiE-rRSS pipeline (SER model when flagged) ----
keys <- c("input_file", "study", "ld")
rss  <- res %>% filter(model == "SuSiE-RSS", !fit_failed)
fit  <- res %>% filter(model == "SuSiE-rRSS", !is_ser, !fit_failed)
ser  <- res %>% filter(model == "SuSiE-rRSS", is_ser)
use_ser <- fit %>% filter(R_reliability_flag %in% TRUE) %>% select(all_of(keys)) %>% semi_join(ser, by = keys)
rrss <- bind_rows(ser %>% semi_join(use_ser, by = keys), fit %>% anti_join(use_ser, by = keys))

set.seed(1)
summ <- bind_rows(rss, rrss) %>%
  group_by(study, ld, match, model, h2g, causal) %>%
  group_modify(~ boot_cell_mix(.x, 1000)) %>%
  ungroup()

pl <- summ %>%
  mutate(ncp = fct_reorder(paste0("NCP = ", formatC(n_GWAS * h2g, format = "f", digits = 0)), h2g)) %>%
  transmute(study, ld, match, model, ncp, causal,
            Power_est = power, Power_lo = power_lo, Power_hi = power_hi,
            FDR_est   = FDR,   FDR_lo   = FDR_lo,   FDR_hi   = FDR_hi) %>%
  pivot_longer(-c(study, ld, match, model, ncp, causal), names_to = c("metric", ".value"), names_sep = "_") %>%
  mutate(metric = factor(metric, levels = c("Power", "FDR")),
         model  = factor(model, levels = c("SuSiE-RSS", "SuSiE-rRSS")),
         match  = factor(match, levels = c("Matched LD", "Cross-population LD")))
save_data(pl, paste0(data_name, "_Power_FDR"))